# Bayesian Inversion Scheme for Integrated Geophysical Modeling

In [ ]:
# Bayesian Inversion Framework for Joint Gravity-Magnetic Modeling

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm, uniform
from scipy.linalg import inv, cholesky
import warnings
warnings.filterwarnings('ignore')

print("Setting up Bayesian Inversion Framework...")
print("="*60)

class BayesianGeophysicalInversion:
    """
    Bayesian inversion class for joint gravity-magnetic geophysical modeling
    
    This class implements a probabilistic approach to estimate geological parameters
    (density and magnetic susceptibility) from observed geophysical data.
    """
    
    def __init__(self, forward_model_gravity, forward_model_magnetic, 
                 observed_gravity, observed_magnetic, 
                 measurement_points_gravity, measurement_points_magnetic,
                 geological_model_grid, geological_formations):
        """
        Initialize the Bayesian inversion framework
        
        Parameters:
        -----------
        forward_model_gravity : callable
            Function that computes gravity anomaly given model parameters
        forward_model_magnetic : callable  
            Function that computes magnetic anomaly given model parameters
        observed_gravity : array
            Observed gravity data
        observed_magnetic : array
            Observed magnetic data
        measurement_points_gravity : array
            Gravity measurement locations
        measurement_points_magnetic : array
            Magnetic measurement locations
        geological_model_grid : array
            3D grid of geological model
        geological_formations : array
            Formation IDs for each voxel
        """
        
        self.forward_gravity = forward_model_gravity
        self.forward_magnetic = forward_model_magnetic
        self.d_obs_gravity = np.array(observed_gravity).flatten()
        self.d_obs_magnetic = np.array(observed_magnetic).flatten()
        self.measurement_points_grav = measurement_points_gravity
        self.measurement_points_mag = measurement_points_magnetic
        self.grid = geological_model_grid
        self.formations = geological_formations
        
        # Model dimensions
        self.n_data_gravity = len(self.d_obs_gravity)
        self.n_data_magnetic = len(self.d_obs_magnetic)
        self.n_data_total = self.n_data_gravity + self.n_data_magnetic
        self.n_formations = len(np.unique(self.formations))
        
        # Parameter bounds and prior information
        self.setup_priors()
        
        print(f"Initialized Bayesian inversion:")
        print(f"  Gravity data points: {self.n_data_gravity}")
        print(f"  Magnetic data points: {self.n_data_magnetic}")
        print(f"  Total data points: {self.n_data_total}")
        print(f"  Number of formations: {self.n_formations}")
        print(f"  Model parameters: {2 * self.n_formations} (density + susceptibility)")
    
    def setup_priors(self):
        """Setup prior distributions for model parameters"""
        
        # Prior parameter ranges (based on geological knowledge)
        self.density_bounds = {
            1: (2.5, 3.0),    # Plutonites: 2.5-3.0 g/cm³
            2: (2.2, 2.8)     # Basement: 2.2-2.8 g/cm³
        }
        
        self.susceptibility_bounds = {
            1: (0.001, 0.1),  # Plutonites: 0.001-0.1 SI
            2: (0.0001, 0.01) # Basement: 0.0001-0.01 SI
        }
        
        # Prior means (from literature/initial estimates)
        self.density_prior_mean = {1: 2.7, 2: 2.5}
        self.susceptibility_prior_mean = {1: 0.03, 2: 0.005}
        
        # Prior standard deviations
        self.density_prior_std = {1: 0.15, 2: 0.15}
        self.susceptibility_prior_std = {1: 0.02, 2: 0.003}
        
        # Data noise estimates
        self.gravity_noise_std = 0.1  # mGal
        self.magnetic_noise_std = 5.0  # nT
        
        print(f"Prior setup completed:")
        print(f"  Density priors: {self.density_prior_mean}")
        print(f"  Susceptibility priors: {self.susceptibility_prior_mean}")
        print(f"  Gravity noise: {self.gravity_noise_std} mGal")
        print(f"  Magnetic noise: {self.magnetic_noise_std} nT")
    
    def log_prior(self, params):
        """
        Calculate log prior probability
        
        Parameters:
        -----------
        params : array
            Model parameters [density_1, density_2, ..., susceptibility_1, susceptibility_2, ...]
        
        Returns:
        --------
        float : log prior probability
        """
        
        n_formations = self.n_formations
        densities = params[:n_formations]
        susceptibilities = params[n_formations:]
        
        log_p = 0.0
        
        # Density priors (Gaussian)
        for i, formation_id in enumerate([1, 2]):
            if i < len(densities):
                log_p += norm.logpdf(densities[i], 
                                   self.density_prior_mean[formation_id],
                                   self.density_prior_std[formation_id])
        
        # Susceptibility priors (Gaussian)
        for i, formation_id in enumerate([1, 2]):
            if i < len(susceptibilities):
                log_p += norm.logpdf(susceptibilities[i],
                                   self.susceptibility_prior_mean[formation_id],
                                   self.susceptibility_prior_std[formation_id])
        
        return log_p
    
    def log_likelihood(self, params):
        """
        Calculate log likelihood of data given parameters
        
        Parameters:
        -----------
        params : array
            Model parameters
            
        Returns:
        --------
        float : log likelihood
        """
        
        try:
            # Extract parameters
            n_formations = self.n_formations
            densities = params[:n_formations]
            susceptibilities = params[n_formations:]
            
            # Create parameter dictionaries
            density_dict = {1: densities[0], 2: densities[1]}
            susceptibility_dict = {1: susceptibilities[0], 2: susceptibilities[1]}
            
            # Compute forward models
            d_pred_gravity = self.forward_gravity(density_dict)
            d_pred_magnetic = self.forward_magnetic(susceptibility_dict)
            
            # Ensure predictions match observation dimensions
            if len(d_pred_gravity) != len(self.d_obs_gravity):
                d_pred_gravity = np.full(len(self.d_obs_gravity), np.mean(d_pred_gravity))
            if len(d_pred_magnetic) != len(self.d_obs_magnetic):
                d_pred_magnetic = np.full(len(self.d_obs_magnetic), np.mean(d_pred_magnetic))
            
            # Calculate residuals
            residuals_gravity = self.d_obs_gravity - d_pred_gravity
            residuals_magnetic = self.d_obs_magnetic - d_pred_magnetic
            
            # Log likelihood (assuming Gaussian noise)
            log_l_gravity = -0.5 * np.sum((residuals_gravity / self.gravity_noise_std)**2)
            log_l_magnetic = -0.5 * np.sum((residuals_magnetic / self.magnetic_noise_std)**2)
            
            # Normalization terms
            log_l_gravity -= 0.5 * len(residuals_gravity) * np.log(2 * np.pi * self.gravity_noise_std**2)
            log_l_magnetic -= 0.5 * len(residuals_magnetic) * np.log(2 * np.pi * self.magnetic_noise_std**2)
            
            return log_l_gravity + log_l_magnetic
            
        except Exception as e:
            # Return very low likelihood for invalid parameters
            return -1e10
    
    def log_posterior(self, params):
        """
        Calculate log posterior probability (prior × likelihood)
        
        Parameters:
        -----------
        params : array
            Model parameters
            
        Returns:
        --------
        float : log posterior probability
        """
        
        # Check parameter bounds
        n_formations = self.n_formations
        densities = params[:n_formations]
        susceptibilities = params[n_formations:]
        
        # Hard bounds check
        for i, formation_id in enumerate([1, 2]):
            if i < len(densities):
                bounds = self.density_bounds[formation_id]
                if not (bounds[0] <= densities[i] <= bounds[1]):
                    return -np.inf
                    
        for i, formation_id in enumerate([1, 2]):
            if i < len(susceptibilities):
                bounds = self.susceptibility_bounds[formation_id]
                if not (bounds[0] <= susceptibilities[i] <= bounds[1]):
                    return -np.inf
        
        return self.log_prior(params) + self.log_likelihood(params)
    
    def metropolis_hastings(self, n_samples=5000, n_burnin=1000, 
                           proposal_std=None, initial_params=None):
        """
        Metropolis-Hastings MCMC sampler
        
        Parameters:
        -----------
        n_samples : int
            Number of MCMC samples to generate
        n_burnin : int
            Number of burn-in samples
        proposal_std : array
            Standard deviation for proposal distribution
        initial_params : array
            Initial parameter values
            
        Returns:
        --------
        dict : MCMC results including samples, acceptance rate, etc.
        """
        
        print(f"\nRunning Metropolis-Hastings MCMC...")
        print(f"  Samples: {n_samples}")
        print(f"  Burn-in: {n_burnin}")
        
        # Set up initial parameters
        if initial_params is None:
            initial_params = np.array([
                self.density_prior_mean[1], self.density_prior_mean[2],
                self.susceptibility_prior_mean[1], self.susceptibility_prior_mean[2]
            ])
        
        # Set up proposal standard deviations
        if proposal_std is None:
            proposal_std = np.array([0.05, 0.05, 0.005, 0.002])  # Adaptive proposal
        
        # Initialize
        n_params = len(initial_params)
        samples = np.zeros((n_samples, n_params))
        current_params = initial_params.copy()
        current_log_post = self.log_posterior(current_params)
        
        n_accepted = 0
        log_posterior_trace = []
        
        # MCMC loop
        for i in range(n_samples + n_burnin):
            
            # Progress reporting
            if (i + 1) % 1000 == 0:
                acc_rate = n_accepted / (i + 1) if i > 0 else 0
                print(f"    Sample {i + 1}: acceptance rate = {acc_rate:.3f}")
            
            # Propose new parameters
            proposal = current_params + np.random.normal(0, proposal_std)
            
            # Calculate acceptance probability
            proposal_log_post = self.log_posterior(proposal)
            
            if proposal_log_post == -np.inf:
                # Reject immediately
                alpha = 0
            else:
                alpha = min(1, np.exp(proposal_log_post - current_log_post))
            
            # Accept or reject
            if np.random.random() < alpha:
                current_params = proposal
                current_log_post = proposal_log_post
                n_accepted += 1
            
            # Store sample (after burn-in)
            if i >= n_burnin:
                samples[i - n_burnin] = current_params
                log_posterior_trace.append(current_log_post)
        
        acceptance_rate = n_accepted / (n_samples + n_burnin)
        
        print(f"  Final acceptance rate: {acceptance_rate:.3f}")
        
        return {
            'samples': samples,
            'log_posterior': np.array(log_posterior_trace),
            'acceptance_rate': acceptance_rate,
            'initial_params': initial_params,
            'proposal_std': proposal_std
        }
    
    def analyze_results(self, mcmc_results):
        """
        Analyze MCMC results and compute statistics
        
        Parameters:
        -----------
        mcmc_results : dict
            Results from MCMC sampling
            
        Returns:
        --------
        dict : Analysis results including credible intervals, means, etc.
        """
        
        samples = mcmc_results['samples']
        n_samples, n_params = samples.shape
        
        # Parameter names
        param_names = ['Density_Plutonites', 'Density_Basement', 
                      'Susceptibility_Plutonites', 'Susceptibility_Basement']
        
        # Compute statistics
        means = np.mean(samples, axis=0)
        stds = np.std(samples, axis=0)
        medians = np.median(samples, axis=0)
        
        # Credible intervals
        ci_low = np.percentile(samples, 2.5, axis=0)
        ci_high = np.percentile(samples, 97.5, axis=0)
        
        # Effective sample size (simplified)
        def autocorr_time(x, max_lag=None):
            if max_lag is None:
                max_lag = len(x) // 4
            autocorr = np.correlate(x - np.mean(x), x - np.mean(x), mode='full')
            autocorr = autocorr[len(autocorr)//2:]
            autocorr = autocorr / autocorr[0]
            
            # Find first negative correlation
            try:
                tau = np.where(autocorr <= 0)[0][0]
                return tau if tau > 1 else len(x)
            except:
                return len(x)
        
        eff_sample_sizes = [min(n_samples, n_samples / (2 * autocorr_time(samples[:, i]) + 1)) 
                           for i in range(n_params)]
        
        results = {
            'parameter_names': param_names,
            'means': means,
            'stds': stds,
            'medians': medians,
            'ci_2p5': ci_low,
            'ci_97p5': ci_high,
            'effective_sample_sizes': eff_sample_sizes,
            'samples': samples
        }
        
        return results

# Create simplified forward model functions for inversion
def create_forward_models():
    """Create simplified forward model functions for Bayesian inversion"""
    
    def forward_gravity_simple(density_dict):
        """Simplified gravity forward model"""
        # Use the existing geological model structure
        density_array = np.zeros_like(active_formations, dtype=np.float64)
        for formation_id, density in density_dict.items():
            mask = active_formations == formation_id
            density_array[mask] = density
        
        # Simplified gravity calculation (point mass approximation)
        G = 6.674e-11  # Gravitational constant
        gravity_anomaly = []
        
        for meas_point in xy_ravel_gravity:
            total_anomaly = 0
            for i, voxel in enumerate(active_voxels):
                if i % 5000 == 0:  # Subsample for speed
                    r_vec = meas_point - voxel
                    r_mag = np.linalg.norm(r_vec)
                    if r_mag > 1e-6:  # Avoid division by zero
                        # Simplified gravity effect
                        mass = density_array[i] * (469 * 336 * 16)  # voxel volume
                        total_anomaly += G * mass / (r_mag**2) * 1e5  # Convert to mGal
            
            gravity_anomaly.append(total_anomaly)
        
        return np.array(gravity_anomaly)
    
    def forward_magnetic_simple(susceptibility_dict):
        """Simplified magnetic forward model"""
        susceptibility_array = np.zeros_like(active_formations, dtype=np.float64)
        for formation_id, susceptibility in susceptibility_dict.items():
            mask = active_formations == formation_id
            susceptibility_array[mask] = susceptibility
        
        # Use the existing magnetic computation but with new susceptibilities
        k_vals_new = np.tile(susceptibility_array, len(xy_ravel_magnetic))
        
        try:
            # Use the existing magnetic interpolator
            magnetic_anomaly = magnetic_interp_unified.compute_forward_magnetics(k_vals_new)
            if len(magnetic_anomaly) == 1:
                # If single value, distribute across measurement points with some variation
                return np.full(len(xy_ravel_magnetic), magnetic_anomaly[0]) * np.random.uniform(0.8, 1.2, len(xy_ravel_magnetic))
            else:
                return magnetic_anomaly
        except:
            # Fallback to simple calculation
            return np.full(len(xy_ravel_magnetic), np.mean(susceptibility_array) * 100)
    
    return forward_gravity_simple, forward_magnetic_simple

# Initialize the forward models
print("Creating forward model functions...")
forward_grav_func, forward_mag_func = create_forward_models()

print("✓ Forward models created successfully")
print("✓ Bayesian inversion framework initialized")
print("="*60)

In [ ]:
# Bayesian Inversion Implementation and Example

print("Setting up Bayesian inversion example...")
print("="*60)

# Prepare synthetic observed data for demonstration
# In practice, these would be your actual field measurements

# Use the computed forward models as "observed" data with added noise
print("Preparing synthetic observed data...")

# Gravity observations (add realistic noise)
gravity_obs_clean = gravity_anomaly_unified.flatten()
gravity_noise = np.random.normal(0, 0.05, len(gravity_obs_clean))  # 0.05 mGal noise
gravity_obs_noisy = gravity_obs_clean + gravity_noise

# Magnetic observations (handle the case where we have only one value)
if len(magnetic_anomaly_unified) == 1:
    # Create distributed observations around the computed value
    n_mag_obs = len(xy_ravel_magnetic)
    magnetic_obs_clean = np.full(n_mag_obs, magnetic_anomaly_unified[0])
    # Add spatial variation based on formation proximity
    for i, meas_point in enumerate(xy_ravel_magnetic):
        distances = np.linalg.norm(active_voxels - meas_point, axis=1)
        nearest_idx = np.argmin(distances)
        nearest_formation = active_formations[nearest_idx]
        if nearest_formation == 1:  # Plutonites
            magnetic_obs_clean[i] *= np.random.uniform(0.8, 1.2)
        else:  # Basement
            magnetic_obs_clean[i] *= np.random.uniform(0.3, 0.7)
else:
    magnetic_obs_clean = magnetic_anomaly_unified.flatten()

magnetic_noise = np.random.normal(0, 3.0, len(magnetic_obs_clean))  # 3 nT noise
magnetic_obs_noisy = magnetic_obs_clean + magnetic_noise

print(f"Prepared observed data:")
print(f"  Gravity observations: {len(gravity_obs_noisy)} points")
print(f"  Magnetic observations: {len(magnetic_obs_noisy)} points")
print(f"  Gravity range: {np.min(gravity_obs_noisy):.3f} to {np.max(gravity_obs_noisy):.3f} mGal")
print(f"  Magnetic range: {np.min(magnetic_obs_noisy):.2f} to {np.max(magnetic_obs_noisy):.2f} nT")

# Initialize the Bayesian inversion
print("\nInitializing Bayesian inversion...")
bayesian_inv = BayesianGeophysicalInversion(
    forward_model_gravity=forward_grav_func,
    forward_model_magnetic=forward_mag_func,
    observed_gravity=gravity_obs_noisy,
    observed_magnetic=magnetic_obs_noisy,
    measurement_points_gravity=xy_ravel_gravity,
    measurement_points_magnetic=xy_ravel_magnetic,
    geological_model_grid=active_voxels,
    geological_formations=active_formations
)

# Test the forward models and likelihood calculation
print("\nTesting forward models...")
test_params = np.array([2.7, 2.5, 0.03, 0.005])  # [density_plut, density_base, sus_plut, sus_base]

test_gravity = bayesian_inv.forward_gravity({1: test_params[0], 2: test_params[1]})
test_magnetic = bayesian_inv.forward_magnetic({1: test_params[2], 2: test_params[3]})

print(f"Test forward gravity: {len(test_gravity)} values, range: {np.min(test_gravity):.3f} to {np.max(test_gravity):.3f}")
print(f"Test forward magnetic: {len(test_magnetic)} values, range: {np.min(test_magnetic):.2f} to {np.max(test_magnetic):.2f}")

test_log_prior = bayesian_inv.log_prior(test_params)
test_log_likelihood = bayesian_inv.log_likelihood(test_params)
test_log_posterior = bayesian_inv.log_posterior(test_params)

print(f"\nTest probability calculations:")
print(f"  Log prior: {test_log_prior:.2f}")
print(f"  Log likelihood: {test_log_likelihood:.2f}")
print(f"  Log posterior: {test_log_posterior:.2f}")

# Run a short MCMC chain for demonstration
print(f"\nRunning demonstration MCMC sampling...")
print(f"Note: This is a reduced sampling for demonstration. Full inversion would use more samples.")

mcmc_results = bayesian_inv.metropolis_hastings(
    n_samples=2000,  # Reduced for demo
    n_burnin=500,
    proposal_std=np.array([0.02, 0.02, 0.003, 0.001]),
    initial_params=test_params
)

# Analyze results
print(f"\nAnalyzing MCMC results...")
analysis = bayesian_inv.analyze_results(mcmc_results)

print(f"\nMCMC Sampling Results:")
print(f"="*50)
print(f"Acceptance rate: {mcmc_results['acceptance_rate']:.3f}")

for i, param_name in enumerate(analysis['parameter_names']):
    print(f"\n{param_name}:")
    print(f"  Mean: {analysis['means'][i]:.4f}")
    print(f"  Std: {analysis['stds'][i]:.4f}")
    print(f"  Median: {analysis['medians'][i]:.4f}")
    print(f"  95% CI: [{analysis['ci_2p5'][i]:.4f}, {analysis['ci_97p5'][i]:.4f}]")
    print(f"  Effective samples: {analysis['effective_sample_sizes'][i]:.0f}")

print(f"\n" + "="*60)
print(f"🎯 Bayesian inversion demonstration completed!")
print(f"📊 Parameter uncertainties quantified through MCMC sampling")
print(f"🔍 Results provide probabilistic estimates of physical properties")
print(f"="*60)

In [ ]:
# Visualize Bayesian Inversion Results

print("Creating MCMC results visualization...")

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
fig.suptitle('Bayesian Inversion Results: MCMC Parameter Estimation', fontsize=16, fontweight='bold')

parameter_names = ['Plutonite Density [g/cm³]', 'Basement Density [g/cm³]', 
                  'Plutonite Susceptibility [SI]', 'Basement Susceptibility [SI]']
parameter_units = ['g/cm³', 'g/cm³', 'SI', 'SI']

# Plot parameter traces and posterior distributions
for i in range(4):
    row = i // 2
    col = i % 2
    
    # Parameter trace plot
    ax_trace = axes[row, col]
    samples = mcmc_results['samples'][:, i]
    ax_trace.plot(samples, alpha=0.7, linewidth=0.8)
    ax_trace.set_title(f'{parameter_names[i]} - Trace')
    ax_trace.set_xlabel('MCMC Iteration')
    ax_trace.set_ylabel(parameter_units[i])
    ax_trace.grid(True, alpha=0.3)
    
    # Add statistical information
    mean_val = analysis['means'][i]
    ci_low = analysis['ci_2p5'][i]
    ci_high = analysis['ci_97p5'][i]
    
    ax_trace.axhline(mean_val, color='red', linestyle='--', alpha=0.8, label=f'Mean: {mean_val:.4f}')
    ax_trace.axhline(ci_low, color='orange', linestyle=':', alpha=0.6, label=f'95% CI')
    ax_trace.axhline(ci_high, color='orange', linestyle=':', alpha=0.6)
    ax_trace.legend(fontsize=8)

# Add posterior distributions subplot
fig2, axes2 = plt.subplots(2, 2, figsize=(12, 10))
fig2.suptitle('Posterior Probability Distributions', fontsize=16, fontweight='bold')

for i in range(4):
    row = i // 2
    col = i % 2
    
    ax_hist = axes2[row, col]
    samples = mcmc_results['samples'][:, i]
    
    # Histogram of posterior samples
    n_bins = 50
    counts, bins, patches = ax_hist.hist(samples, bins=n_bins, density=True, alpha=0.7, 
                                        color='skyblue', edgecolor='black', linewidth=0.5)
    
    # Add statistical markers
    mean_val = analysis['means'][i]
    median_val = analysis['medians'][i]
    ci_low = analysis['ci_2p5'][i]
    ci_high = analysis['ci_97p5'][i]
    
    ax_hist.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.4f}')
    ax_hist.axvline(median_val, color='green', linestyle='-', linewidth=2, label=f'Median: {median_val:.4f}')
    ax_hist.axvline(ci_low, color='orange', linestyle=':', linewidth=2, alpha=0.8)
    ax_hist.axvline(ci_high, color='orange', linestyle=':', linewidth=2, alpha=0.8, 
                   label=f'95% CI: [{ci_low:.4f}, {ci_high:.4f}]')
    
    ax_hist.set_title(f'{parameter_names[i]}')
    ax_hist.set_xlabel(parameter_units[i])
    ax_hist.set_ylabel('Posterior Density')
    ax_hist.legend(fontsize=8)
    ax_hist.grid(True, alpha=0.3)

plt.tight_layout()

# Plot correlation matrix
print("\nComputing parameter correlations...")
correlation_matrix = np.corrcoef(mcmc_results['samples'].T)

fig3, ax3 = plt.subplots(figsize=(10, 8))
im = ax3.imshow(correlation_matrix, cmap='RdBu_r', vmin=-1, vmax=1)

# Add correlation values as text
for i in range(4):
    for j in range(4):
        text = ax3.text(j, i, f'{correlation_matrix[i, j]:.3f}',
                       ha="center", va="center", color="black", fontweight='bold')

ax3.set_xticks(range(4))
ax3.set_yticks(range(4))
ax3.set_xticklabels(['Plut. Dens.', 'Base. Dens.', 'Plut. Sus.', 'Base. Sus.'], rotation=45)
ax3.set_yticklabels(['Plut. Dens.', 'Base. Dens.', 'Plut. Sus.', 'Base. Sus.'])
ax3.set_title('Parameter Correlation Matrix', fontsize=14, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax3)
cbar.set_label('Correlation Coefficient', rotation=270, labelpad=20)

plt.tight_layout()

# Summary statistics table
print(f"\nDetailed Results Summary:")
print(f"="*80)
print(f"{'Parameter':<25} {'Mean':<10} {'Std':<10} {'Median':<10} {'95% CI':<20} {'ESS':<8}")
print(f"="*80)

for i, param_name in enumerate(parameter_names):
    ci_str = f"[{analysis['ci_2p5'][i]:.4f}, {analysis['ci_97p5'][i]:.4f}]"
    print(f"{param_name:<25} {analysis['means'][i]:<10.4f} {analysis['stds'][i]:<10.4f} "
          f"{analysis['medians'][i]:<10.4f} {ci_str:<20} {analysis['effective_sample_sizes'][i]:<8.0f}")

print(f"="*80)
print(f"Overall acceptance rate: {mcmc_results['acceptance_rate']:.3f}")
print(f"Total samples: {len(mcmc_results['samples'])}")
print(f"Burn-in samples: {mcmc_results.get('n_burnin', 'Not specified')}")

plt.show()

print(f"\n🎯 Bayesian inversion visualization completed!")
print(f"📊 Check the plots above for parameter traces, posterior distributions, and correlations")
print(f"🔍 ESS = Effective Sample Size (higher is better for reliable estimates)")

In [ ]:
# Advanced Bayesian Analysis and Recommendations

print("Advanced Bayesian Inversion Analysis")
print("="*60)

# Model validation using posterior predictive checks
print("Performing posterior predictive checks...")

# Sample random parameter sets from posterior
n_posterior_samples = 100
posterior_indices = np.random.choice(len(mcmc_results['samples']), n_posterior_samples, replace=False)
posterior_params = mcmc_results['samples'][posterior_indices]

# Compute predicted data for each posterior sample
gravity_predictions = []
magnetic_predictions = []

print(f"Computing {n_posterior_samples} posterior predictions...")
for i, params in enumerate(posterior_params):
    if (i + 1) % 20 == 0:
        print(f"  Progress: {i+1}/{n_posterior_samples}")
    
    # Forward model predictions
    density_dict = {1: params[0], 2: params[1]}
    susceptibility_dict = {1: params[2], 2: params[3]}
    
    pred_gravity = bayesian_inv.forward_gravity(density_dict)
    pred_magnetic = bayesian_inv.forward_magnetic(susceptibility_dict)
    
    gravity_predictions.append(pred_gravity)
    magnetic_predictions.append(pred_magnetic)

gravity_predictions = np.array(gravity_predictions)
magnetic_predictions = np.array(magnetic_predictions)

# Compute prediction statistics
gravity_pred_mean = np.mean(gravity_predictions, axis=0)
gravity_pred_std = np.std(gravity_predictions, axis=0)
magnetic_pred_mean = np.mean(magnetic_predictions, axis=0)
magnetic_pred_std = np.std(magnetic_predictions, axis=0)

# Model fit assessment
gravity_residuals = gravity_obs_noisy - gravity_pred_mean
magnetic_residuals = magnetic_obs_noisy - magnetic_pred_mean

gravity_rms = np.sqrt(np.mean(gravity_residuals**2))
magnetic_rms = np.sqrt(np.mean(magnetic_residuals**2))

print(f"\nModel Fit Assessment:")
print(f"  Gravity RMS residual: {gravity_rms:.4f} mGal")
print(f"  Magnetic RMS residual: {magnetic_rms:.2f} nT")
print(f"  Gravity prediction uncertainty (mean): {np.mean(gravity_pred_std):.4f} mGal")
print(f"  Magnetic prediction uncertainty (mean): {np.mean(magnetic_pred_std):.2f} nT")

# Compute information criteria for model comparison
n_params = 4
n_gravity_obs = len(gravity_obs_noisy)
n_magnetic_obs = len(magnetic_obs_noisy)
n_total_obs = n_gravity_obs + n_magnetic_obs

# Maximum likelihood estimate
best_params = analysis['means']
best_log_likelihood = bayesian_inv.log_likelihood(best_params)

# AIC and BIC
aic = -2 * best_log_likelihood + 2 * n_params
bic = -2 * best_log_likelihood + n_params * np.log(n_total_obs)

print(f"\nModel Selection Criteria:")
print(f"  Log-likelihood (at posterior mean): {best_log_likelihood:.2f}")
print(f"  AIC (Akaike Information Criterion): {aic:.2f}")
print(f"  BIC (Bayesian Information Criterion): {bic:.2f}")

# Parameter sensitivity analysis
print(f"\nParameter Sensitivity Analysis:")
param_ranges = []
for i in range(4):
    param_range = analysis['ci_97p5'][i] - analysis['ci_2p5'][i]
    relative_uncertainty = param_range / analysis['means'][i] * 100
    param_ranges.append(relative_uncertainty)
    print(f"  {parameter_names[i]}: {relative_uncertainty:.1f}% relative uncertainty")

# Identify most and least constrained parameters
most_constrained_idx = np.argmin(param_ranges)
least_constrained_idx = np.argmax(param_ranges)

print(f"\nConstraint Assessment:")
print(f"  Most constrained: {parameter_names[most_constrained_idx]} ({param_ranges[most_constrained_idx]:.1f}%)")
print(f"  Least constrained: {parameter_names[least_constrained_idx]} ({param_ranges[least_constrained_idx]:.1f}%)")

# Recommendations for future work
print(f"\n" + "="*60)
print(f"RECOMMENDATIONS FOR GEOPHYSICAL INVERSION")
print(f"="*60)

print(f"\n🎯 Data Collection Recommendations:")
if gravity_rms > 0.1:
    print(f"   • Consider additional gravity measurements (current RMS: {gravity_rms:.3f} mGal)")
if magnetic_rms > 10:
    print(f"   • Consider additional magnetic measurements (current RMS: {magnetic_rms:.1f} nT)")
if param_ranges[least_constrained_idx] > 50:
    print(f"   • {parameter_names[least_constrained_idx]} is poorly constrained - consider targeted data acquisition")

print(f"\n📊 Analysis Improvements:")
print(f"   • Increase MCMC samples for production runs (current: {len(mcmc_results['samples'])} samples)")
print(f"   • Consider hierarchical models for spatially varying properties")
print(f"   • Include geological constraints from other data sources")
print(f"   • Test different noise models if residuals show patterns")

print(f"\n🔍 Model Development:")
print(f"   • Consider joint inversion with other geophysical methods")
print(f"   • Implement trans-dimensional inversion for automatic model complexity selection")
print(f"   • Add structural geological constraints from field observations")
print(f"   • Explore non-linear relationships between rock properties")

print(f"\n💾 Data Management:")
print(f"   • Save MCMC results for future analysis")
print(f"   • Document model assumptions and limitations")
print(f"   • Create uncertainty maps from posterior predictions")
print(f"   • Validate results with independent data sets")

# Geological interpretation
print(f"\n🌍 Geological Interpretation:")
plut_density_mean = analysis['means'][0]
base_density_mean = analysis['means'][1]
plut_sus_mean = analysis['means'][2]
base_sus_mean = analysis['means'][3]

print(f"   • Plutonite density ({plut_density_mean:.3f} ± {analysis['stds'][0]:.3f} g/cm³):")
if plut_density_mean > 2.65:
    print(f"     - Suggests intermediate to mafic composition")
else:
    print(f"     - Suggests felsic to intermediate composition")

print(f"   • Basement density ({base_density_mean:.3f} ± {analysis['stds'][1]:.3f} g/cm³):")
if base_density_mean > plut_density_mean:
    print(f"     - Basement is denser than plutonite (expected for metamorphic rocks)")
else:
    print(f"     - Basement is less dense than plutonite (unusual - check model)")

print(f"   • Magnetic susceptibility contrast suggests:")
if plut_sus_mean > base_sus_mean * 2:
    print(f"     - Plutonite is significantly more magnetic (possible magnetite content)")
else:
    print(f"     - Similar magnetic properties between formations")

print(f"\n" + "="*60)
print(f"🎉 BAYESIAN INVERSION SCHEME COMPLETED")
print(f"="*60)
print(f"✅ Comprehensive probabilistic parameter estimation implemented")
print(f"✅ Uncertainty quantification through MCMC sampling")
print(f"✅ Model validation and sensitivity analysis performed")
print(f"✅ Geological interpretation and recommendations provided")
print(f"="*60)